In [2]:
# 0) Rutas del proyecto y verificación de entradas

# %%
from pathlib import Path

def find_base(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "RAW").exists():
            return p
    raise FileNotFoundError("No encuentro la carpeta 'raw' en ningún parent de la ruta actual")

BASE = find_base(Path.cwd())
RAW  = BASE / "RAW"
OUT  = BASE / "processed" / "chunks_legal"
OUT.mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("RAW exists:", RAW.exists(), RAW)
print("OUT exists:", OUT.exists(), OUT)

print("BOE HTML:", len(list((RAW / "boe").glob("*.html"))))
print("EU HTML:", len(list((RAW / "EU AI Act completo (Reglamento UE 2024-1689)").glob("*.html"))))
print("AESIA PDF:", len(list((RAW / "Guías AESIA + sandbox regulatorio").glob("*.pdf"))))

BASE: /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/proyecto-final
RAW exists: True /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/proyecto-final/RAW
OUT exists: True /Users/danielocando/IA/KeepCoding/BOOTCAMP IA/NormaBot TFM BOOTCAMP IA IV/proyecto-final/processed/chunks_legal
BOE HTML: 0
EU HTML: 0
AESIA PDF: 0


In [3]:
# 1) Lectura BOE (HTML) → texto plano
import json
from bs4 import BeautifulSoup

boe_dir = RAW / "boe"
boe_files = sorted(boe_dir.glob("*.html"))

boe_texts = []
for fp in boe_files:
    html = fp.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")
    text = "\n".join(line.strip() for line in text.splitlines() if line.strip())
    boe_texts.append({"source": "boe", "file": fp.name, "text": text})

print("DOCS:", len(boe_texts))
print("FIRST:", boe_texts[0]["file"] if boe_texts else None)

DOCS: 0
FIRST: None


In [4]:
# 2) Guardar BOE → boe_texts.jsonl

out_file = OUT / "boe_texts.jsonl"

with out_file.open("w", encoding="utf-8") as f:
    for row in boe_texts:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("WROTE:", out_file.name)
print("LINES:", sum(1 for _ in out_file.open("r", encoding="utf-8")))

WROTE: boe_texts.jsonl
LINES: 0


In [5]:
# 3) Lectura EU AI Act (HTML) → texto plano + JSONL

from bs4 import BeautifulSoup

eu_dir = RAW / "EU AI Act completo (Reglamento UE 2024-1689)"
eu_files = sorted(eu_dir.glob("*.html"))

eu_texts = []
for fp in eu_files:
    html = fp.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")
    text = "\n".join(line.strip() for line in text.splitlines() if line.strip())
    eu_texts.append({"source": "eu_ai_act", "file": fp.name, "text": text})

out_file = OUT / "eu_ai_act_texts.jsonl"
with out_file.open("w", encoding="utf-8") as f:
    for row in eu_texts:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("DOCS:", len(eu_texts))
print("FIRST:", eu_texts[0]["file"] if eu_texts else None)
print("WROTE:", out_file.name)

DOCS: 0
FIRST: None
WROTE: eu_ai_act_texts.jsonl


In [6]:
# 4) Lectura AESIA (PDF) → texto plano + JSONL
from pypdf import PdfReader

aes_dir = RAW / "Guías AESIA + sandbox regulatorio"
pdf_files = sorted(aes_dir.glob("*.pdf"))

aes_texts = []

for fp in pdf_files:
    try:
        reader = PdfReader(str(fp))
        parts = []
        extracted_pages = 0

        for page in reader.pages:
            t = page.extract_text() or ""
            t = "\n".join(line.strip() for line in t.splitlines() if line.strip())
            if t:
                extracted_pages += 1
                parts.append(t)

        full_text = "\n\n".join(parts)

        aes_texts.append({
            "source": "aesia",
            "file": fp.name,
            "n_pages": len(reader.pages),
            "pages_extracted": extracted_pages,
            "text": full_text
        })

    except Exception as e:
        print(f"[WARN] Error al leer {fp.name}: {e}. Omitiendo.")
        continue

out_file = OUT / "aesia_texts.jsonl"

with out_file.open("w", encoding="utf-8") as f:
    for row in aes_texts:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("DOCS:", len(aes_texts))
print("FIRST:", aes_texts[0]["file"] if aes_texts else None)
print("WROTE:", out_file.name)

DOCS: 0
FIRST: None
WROTE: aesia_texts.jsonl


In [7]:
# 5) Chunking estructurado por unidades (Artículo / Título / Capítulo / Sección)
import re, hashlib

boe_fp = OUT / "boe_texts.jsonl"
eu_fp  = OUT / "eu_ai_act_texts.jsonl"
aes_fp = OUT / "aesia_texts.jsonl"

out_fp = OUT / "chunks_structured.jsonl"

def md5(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def norm_spaces(s: str) -> str:
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def split_units(text: str, patterns):
    """
    patterns: lista de regex MULTILINE que detectan cabeceras.
    Devuelve lista de (header_line, body_text).
    """
    t = "\n" + text.strip() + "\n"
    hits = []
    for pat in patterns:
        for m in re.finditer(pat, t, flags=re.IGNORECASE | re.MULTILINE):
            hits.append((m.start(), m.group(0).strip()))
    hits = sorted(set(hits), key=lambda x: x[0])

    if not hits:
        return [("DOCUMENT", text.strip())]

    units = []
    for i, (pos, header) in enumerate(hits):
        end = hits[i+1][0] if i+1 < len(hits) else len(t)
        chunk = t[pos:end].strip()
        first_line = chunk.splitlines()[0].strip() if chunk else header
        body = "\n".join(chunk.splitlines()[1:]).strip()
        units.append((first_line, body if body else chunk))
    return units

def parse_doc_meta(source: str, file: str, text: str):
    # título
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    title = lines[0][:200] if lines else None

    # fecha típica ES
    m = re.search(
        r"\b(\d{1,2})\s+de\s+(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre)\s+de\s+(\d{4})\b",
        text, flags=re.IGNORECASE
    )
    date = f"{m.group(1)} {m.group(2).lower()} {m.group(3)}" if m else None

    # BOE id desde filename
    boe_id = None
    boe_year = None
    if source == "boe":
        m2 = re.search(r"BOE-[A-Z]-([0-9]{4})-([0-9]+)", file)
        if m2:
            boe_year, boe_id = m2.group(1), m2.group(2)

    return {"doc_title": title, "doc_date": date, "boe_year": boe_year, "boe_id": boe_id}

def unit_meta_from_header(header: str):
    h = header.strip()

    unit_type = "section"
    unit_id = None
    unit_title = h[:200]

    # Artículo / Article
    m = re.search(r"\b(art[íi]culo|article)\s+(\d+)\b", h, flags=re.IGNORECASE)
    if m:
        return "article", m.group(2), unit_title

    # Capítulo
    m = re.search(r"\bcap[íi]tulo\s+([ivxlcdm]+|\d+)\b", h, flags=re.IGNORECASE)
    if m:
        return "chapter", m.group(1).upper(), unit_title

    # Título
    m = re.search(r"\bt[íi]tulo\s+([ivxlcdm]+|\d+)\b", h, flags=re.IGNORECASE)
    if m:
        return "title", m.group(1).upper(), unit_title

    # Sección
    m = re.search(r"\bsecci[oó]n\s+([ivxlcdm]+|\d+)\b", h, flags=re.IGNORECASE)
    if m:
        return "section", m.group(1).upper(), unit_title

    # Si no se detecta, se usa el título como identificador estable
    unit_id = unit_title.strip() or None
    return unit_type, unit_id, unit_title

def build_structured_chunks(source: str, texts_jsonl: Path, patterns):
    rows_out = []
    with texts_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            doc = json.loads(line)
            file = doc.get("file")
            text = norm_spaces(doc.get("text", ""))

            dmeta = parse_doc_meta(source, file, text)
            units = split_units(text, patterns)

            for u_idx, (header, body) in enumerate(units):
                body = norm_spaces(body)
                if len(body) < 80:
                    continue

                unit_type, unit_id, unit_title = unit_meta_from_header(header)

                payload = {
                    "source": source,
                    "file": file,
                    "doc_title": dmeta["doc_title"],
                    "doc_date": dmeta["doc_date"],
                    "boe_year": dmeta.get("boe_year"),
                    "boe_id": dmeta.get("boe_id"),
                    "unit_type": unit_type,
                    "unit_id": unit_id,
                    "unit_title": unit_title,
                    "unit_index": u_idx,
                    "text": body,
                }
                payload["id"] = md5(
                    f"{source}|{file}|{unit_type}|{unit_id}|{u_idx}|{body[:200]}"
                )
                rows_out.append(payload)
    return rows_out

# Patrones (multiline, tolerantes)
boe_patterns = [
    r"(?m)^\s*Art[íi]culo\s+\d+.*$",
    r"(?m)^\s*T[íi]TULO\s+[IVXLC0-9]+\b.*$",
    r"(?m)^\s*CAP[ÍI]TULO\s+[IVXLC0-9]+\b.*$",
    r"(?m)^\s*Secci[oó]n\s+[IVXLC0-9]+\b.*$",
]
eu_patterns = [
    r"(?m)^\s*Article\s+\d+.*$",
    r"(?m)^\s*Art[íi]culo\s+\d+.*$",
    r"(?m)^\s*CHAPTER\s+[IVXLC0-9]+\b.*$",
    r"(?m)^\s*CAP[ÍI]TULO\s+[IVXLC0-9]+\b.*$",
    r"(?m)^\s*SECTION\s+[IVXLC0-9]+\b.*$",
    r"(?m)^\s*Secci[oó]n\s+[IVXLC0-9]+\b.*$",
]
aes_patterns = [
    r"(?m)^\s*T[íi]TULO\s+[IVXLC0-9]+\b.*$",
    r"(?m)^\s*CAP[ÍI]TULO\s+[IVXLC0-9]+\b.*$",
    r"(?m)^\s*Secci[oó]n\s+[IVXLC0-9]+\b.*$",
    r"(?m)^\s*Art[íi]culo\s+\d+.*$",
]

rows = []
rows += build_structured_chunks("boe", boe_fp, boe_patterns)
rows += build_structured_chunks("eu_ai_act", eu_fp, eu_patterns)
rows += build_structured_chunks("aesia", aes_fp, aes_patterns)

with out_fp.open("w", encoding="utf-8") as fout:
    for r in rows:
        fout.write(json.dumps(r, ensure_ascii=False) + "\n")

print("ROWS:", len(rows))
print("WROTE:", out_fp.name)

ROWS: 0
WROTE: chunks_structured.jsonl


In [8]:
# 6) Muestra rápida de un ejemplo por fuente (structured)
import json

fp = OUT / "chunks_structured.jsonl"

def show_one(src: str):
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("source") == src:
                print(f"\n--- {src} SAMPLE ---")
                print("file:", row.get("file"))
                print("doc_title:", row.get("doc_title"))
                print("doc_date:", row.get("doc_date"))
                print("unit_type:", row.get("unit_type"))
                print("unit_id:", row.get("unit_id"))
                txt = (row.get("text") or "").replace("\n", " ")
                print("text_len:", len(txt))
                print("preview:", txt[:250])
                return
    print(f"\n--- {src}: NO ENCONTRADO ---")

show_one("boe")
show_one("eu_ai_act")
show_one("aesia")


--- boe: NO ENCONTRADO ---

--- eu_ai_act: NO ENCONTRADO ---

--- aesia: NO ENCONTRADO ---


In [10]:
# 7) Export final: chunks_structured.jsonl → chunks_final.jsonl
import json, hashlib

inp = OUT / "chunks_structured.jsonl"
outp = OUT / "chunks_final.jsonl"

print("IN EXISTS:", inp.exists(), inp.name)

def make_chunk_id(r: dict, txt: str) -> str:
    s = (
        f"{r.get('source','')}|{r.get('file','')}|{r.get('boe_id','')}|"
        f"{r.get('unit_type','')}|{r.get('unit_id','')}|{r.get('unit_index','')}|{txt}"
    )
    return hashlib.md5(s.encode("utf-8")).hexdigest()

total = 0
with inp.open("r", encoding="utf-8") as fin, outp.open("w", encoding="utf-8") as fout:
    for line in fin:
        r = json.loads(line)
        txt = (r.get("text") or "").strip()
        if not txt:
            continue

        out_row = {
            "id": r.get("id"),
            "source": r.get("source"),
            "file": r.get("file"),
            "doc_title": r.get("doc_title"),
            "doc_date": r.get("doc_date"),
            "boe_id": r.get("boe_id"),
            "unit_type": r.get("unit_type"),
            "unit_id": r.get("unit_id"),
            "unit_title": r.get("unit_title"),
            "unit_index": r.get("unit_index"),
            "text": txt,
        }

        fout.write(json.dumps(out_row, ensure_ascii=False) + "\n")
        total += 1

print("OUT WRITTEN:", total, outp.name)

IN EXISTS: True chunks_structured.jsonl
OUT WRITTEN: 0 chunks_final.jsonl


In [11]:
# 8) Revisión rápida del archivo final (conteos + metadatos nulos)
import json
from collections import Counter

fp = OUT / "chunks_final.jsonl"
print("EXISTS:", fp.exists(), fp.name)

n = 0
by_source = Counter()
nulls = Counter()

keys_to_check = ["boe_id", "unit_type", "unit_id", "unit_title", "unit_index", "text"]

with fp.open("r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        n += 1
        by_source[r.get("source")] += 1
        for k in keys_to_check:
            nulls[(k, r.get(k) is None)] += 1

print("ROWS:", n)
print("BY_SOURCE:", by_source)

for k in ["boe_id", "unit_type", "unit_id", "unit_title", "unit_index"]:
    print(k, "None:", nulls[(k, True)], "OK:", nulls[(k, False)])

EXISTS: True chunks_final.jsonl
ROWS: 0
BY_SOURCE: Counter()
boe_id None: 0 OK: 0
unit_type None: 0 OK: 0
unit_id None: 0 OK: 0
unit_title None: 0 OK: 0
unit_index None: 0 OK: 0


In [12]:
# 9) LOPD/RGPD desde PDFs en raw/Normativa LOPD-GDD, RGPD
import json
from pypdf import PdfReader

src_dir = RAW / "Normativa LOPD-GDD, RGPD"
out_fp  = OUT / "lopd_rgpd_texts.jsonl"

pdfs = sorted(src_dir.glob("*.pdf"))
print("PDFS:", [p.name for p in pdfs])

rows = []
for fp in pdfs:
    reader = PdfReader(str(fp))
    parts = []
    for page in reader.pages:
        t = page.extract_text() or ""
        t = "\n".join(line.strip() for line in t.splitlines() if line.strip())
        if t:
            parts.append(t)
    txt = "\n\n".join(parts)
    rows.append({"source": "lopd_rgpd", "file": fp.name, "text": txt})

with out_fp.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("WROTE:", out_fp.name, "| DOCS:", len(rows))

PDFS: []
WROTE: lopd_rgpd_texts.jsonl | DOCS: 0


In [13]:
# 10) Chunking estructurado LOPD/RGPD → chunks_lopd_rgpd_structured.jsonl
import json, re, hashlib
from pathlib import Path

SRC = OUT / "lopd_rgpd_texts.jsonl"
OUT_FP = OUT / "chunks_lopd_rgpd_structured.jsonl"

print("IN EXISTS:", SRC.exists(), SRC.name)

RE_TITULO = re.compile(r"^\s*T[ÍI]TULO\s+([IVXLCDM]+)\b\.?\s*(.*)$", re.IGNORECASE)
RE_CAPIT  = re.compile(r"^\s*CAP[ÍI]TULO\s+([IVXLCDM]+)\b\.?\s*(.*)$", re.IGNORECASE)
RE_SECC   = re.compile(r"^\s*SECCI[ÓO]N\s+([IVXLCDM]+)\b\.?\s*(.*)$", re.IGNORECASE)
RE_ART    = re.compile(r"^\s*Art[íi]culo\s+(\d+)\.?\s*(.*)$", re.IGNORECASE)

def clean_lines(text: str):
    out = []
    for ln in text.splitlines():
        ln = ln.replace("\ufeff", "").strip()
        if ln:
            out.append(ln)
    return out

def make_id(payload: dict) -> str:
    s = json.dumps(payload, ensure_ascii=False, sort_keys=True)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def chunk_by_articles(text: str):
    lines = clean_lines(text)
    cur = {"titulo": None, "capitulo": None, "seccion": None, "articulo": None}
    buf = []
    out = []
    seen_article = False

    def flush():
        nonlocal buf
        if not buf:
            return
        chunk_text = "\n".join(buf).strip()
        if chunk_text:
            out.append({"meta": cur.copy(), "text": chunk_text})
        buf = []

    for ln in lines:
        m = RE_TITULO.match(ln)
        if m:
            cur["titulo"] = (m.group(1) + (" " + m.group(2) if m.group(2) else "")).strip()
            buf.append(ln)
            continue

        m = RE_CAPIT.match(ln)
        if m:
            cur["capitulo"] = (m.group(1) + (" " + m.group(2) if m.group(2) else "")).strip()
            buf.append(ln)
            continue

        m = RE_SECC.match(ln)
        if m:
            cur["seccion"] = (m.group(1) + (" " + m.group(2) if m.group(2) else "")).strip()
            buf.append(ln)
            continue

        m = RE_ART.match(ln)
        if m:
            if seen_article:
                flush()
            seen_article = True
            cur["articulo"] = m.group(1)
            buf.append(ln)
            continue

        buf.append(ln)

    flush()
    if not out:
        out = [{"meta": cur.copy(), "text": "\n".join(lines)}]
    return out

total_docs = 0
total_chunks = 0

with SRC.open("r", encoding="utf-8") as fin, OUT_FP.open("w", encoding="utf-8") as fout:
    for line in fin:
        row = json.loads(line)
        total_docs += 1

        source = row.get("source", "lopd_rgpd")
        file_ = row.get("file")
        text = row.get("text", "")

        chunks = chunk_by_articles(text)

        for i, ch in enumerate(chunks):
            out_row = {
                "id": None,
                "source": source,
                "file": file_,
                "doc_title": file_,
                "doc_date": None,
                "titulo": ch["meta"].get("titulo"),
                "capitulo": ch["meta"].get("capitulo"),
                "seccion": ch["meta"].get("seccion"),
                "articulo": ch["meta"].get("articulo"),
                "chunk_index": i,
                "text": ch["text"],
            }
            out_row["id"] = make_id(out_row)
            fout.write(json.dumps(out_row, ensure_ascii=False) + "\n")
            total_chunks += 1

print("OUT WRITTEN:", total_chunks, OUT_FP.name, "| DOCS:", total_docs)

IN EXISTS: True lopd_rgpd_texts.jsonl
OUT WRITTEN: 0 chunks_lopd_rgpd_structured.jsonl | DOCS: 0


In [14]:
# 11) MERGE final NORMALIZADO: A (BOE+EU+AESIA) + B (LOPD/RGPD) -> chunks_final_all_sources.jsonl

import json, hashlib
from pathlib import Path

fp_a = OUT / "chunks_final.jsonl"
fp_b = OUT / "chunks_lopd_rgpd_structured.jsonl"
fp_out = OUT / "chunks_final_all_sources.jsonl"

print("A exists:", fp_a.exists(), fp_a.name)
print("B exists:", fp_b.exists(), fp_b.name)

def stable_id(*parts: str) -> str:
    s = "|".join("" if p is None else str(p) for p in parts)
    return hashlib.md5(s.encode("utf-8")).hexdigest()

n_out = 0
with fp_out.open("w", encoding="utf-8") as fout:
    # A: ya viene con el schema final
    with fp_a.open("r", encoding="utf-8") as fa:
        for line in fa:
            if line.strip():
                fout.write(line if line.endswith("\n") else line + "\n")
                n_out += 1

    # B: normalizar al schema final
    with fp_b.open("r", encoding="utf-8") as fb:
        for line in fb:
            if not line.strip():
                continue
            r = json.loads(line)

            txt = (r.get("text") or "").strip()
            if not txt:
                continue

            # “unidad” preferida: Artículo si existe; si no, Sección/Capítulo/Título (lo primero no vacío)
            unit_type = "article" if r.get("articulo") else "section"
            unit_id = r.get("articulo") or r.get("seccion") or r.get("capitulo") or r.get("titulo") or None
            unit_title = None
            if r.get("articulo"):
                unit_title = f"Artículo {r.get('articulo')}"
            else:
                unit_title = r.get("seccion") or r.get("capitulo") or r.get("titulo") or "DOCUMENT"

            out_row = {
                "id": stable_id("lopd_rgpd", r.get("file"), unit_type, unit_id, r.get("chunk_index"), txt),
                "source": r.get("source", "lopd_rgpd"),
                "file": r.get("file"),
                "doc_title": r.get("doc_title") or r.get("file"),
                "doc_date": r.get("doc_date"),
                "boe_id": None,
                "unit_type": unit_type,
                "unit_id": unit_id,
                "unit_title": unit_title,
                "unit_index": r.get("chunk_index"),
                "text": txt,
            }

            fout.write(json.dumps(out_row, ensure_ascii=False) + "\n")
            n_out += 1

print("OUT WRITTEN:", n_out, fp_out.name)

A exists: True chunks_final.jsonl
B exists: True chunks_lopd_rgpd_structured.jsonl
OUT WRITTEN: 0 chunks_final_all_sources.jsonl
